# Final Comparison - All Three Diseases

Results compiled from notebooks 04 (Heart), 05 (Diabetes), and 06 (Kidney) after training.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

heart_results = {
    'Random Forest':       {'Accuracy': 0.847826, 'Precision': 0.830357, 'Recall': 0.911765, 'F1': 0.869159, 'ROC-AUC': 0.916786},
    'ANN (ReLU)':           {'Accuracy': 0.842391, 'Precision': 0.834862, 'Recall': 0.892157, 'F1': 0.862559, 'ROC-AUC': 0.919536},
    'Logistic Regression':  {'Accuracy': 0.842391, 'Precision': 0.841121, 'Recall': 0.882353, 'F1': 0.861244, 'ROC-AUC': 0.903156},
    'XGBoost':              {'Accuracy': 0.831522, 'Precision': 0.844660, 'Recall': 0.852941, 'F1': 0.848780, 'ROC-AUC': 0.910330},
}

diabetes_results = {
    'XGBoost':              {'Accuracy': 0.759740, 'Precision': 0.673469, 'Recall': 0.611111, 'F1': 0.640777, 'ROC-AUC': 0.827037},
    'Random Forest':        {'Accuracy': 0.733766, 'Precision': 0.651163, 'Recall': 0.518519, 'F1': 0.577320, 'ROC-AUC': 0.811481},
    'ANN (ReLU)':           {'Accuracy': 0.701299, 'Precision': 0.576923, 'Recall': 0.555556, 'F1': 0.566038, 'ROC-AUC': 0.807963},
    'Logistic Regression':  {'Accuracy': 0.707792, 'Precision': 0.600000, 'Recall': 0.500000, 'F1': 0.545455, 'ROC-AUC': 0.812963},
}

kidney_results = {
    'Random Forest':        {'Accuracy': 1.0000, 'Precision': 1.00, 'Recall': 1.00, 'F1': 1.000000, 'ROC-AUC': 1.0},
    'Logistic Regression':  {'Accuracy': 0.9875, 'Precision': 1.00, 'Recall': 0.98, 'F1': 0.989899, 'ROC-AUC': 1.0},
    'XGBoost':              {'Accuracy': 0.9875, 'Precision': 1.00, 'Recall': 0.98, 'F1': 0.989899, 'ROC-AUC': 1.0},
    'ANN (ReLU)':           {'Accuracy': 0.9875, 'Precision': 1.00, 'Recall': 0.98, 'F1': 0.989899, 'ROC-AUC': 1.0},
}

## Combine Into One Table

In [ ]:
heart_df = pd.DataFrame(heart_results).T
heart_df['Disease'] = 'Heart'

diabetes_df = pd.DataFrame(diabetes_results).T
diabetes_df['Disease'] = 'Diabetes'

kidney_df = pd.DataFrame(kidney_results).T
kidney_df['Disease'] = 'Kidney'

combined = pd.concat([heart_df, diabetes_df, kidney_df])
combined.index.name = 'Model'
combined = combined.reset_index()
combined

## Visual Comparison - F1 Score Across Diseases and Models

In [ ]:
pivot = combined.pivot(index='Model', columns='Disease', values='F1')
pivot.plot(kind='bar', figsize=(10, 6))
plt.title('F1 Score Comparison Across Diseases')
plt.ylabel('F1 Score')
plt.ylim(0, 1.05)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Visual Comparison - ROC-AUC Across Diseases and Models

In [ ]:
pivot_auc = combined.pivot(index='Model', columns='Disease', values='ROC-AUC')
pivot_auc.plot(kind='bar', figsize=(10, 6))
plt.title('ROC-AUC Comparison Across Diseases')
plt.ylabel('ROC-AUC')
plt.ylim(0, 1.05)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Best Model Per Disease

In [ ]:
for disease, group in combined.groupby('Disease'):
    best = group.loc[group['F1'].idxmax()]
    print(f"{disease}: Best model = {best['Model']} (F1 = {best['F1']:.4f}, ROC-AUC = {best['ROC-AUC']:.4f})")

## Key Findings

**Heart Disease**: All four models performed close together (F1 in the 0.85-0.87 range). Random Forest had the best F1, ANN (ReLU) had the best ROC-AUC. This is typical for medium-sized, moderately clean tabular data - classical ML and deep learning end up comparable, since a 3-layer ANN doesn't have enough data here to exploit any real advantage over tree-based models.

**Diabetes**: Noticeably lower scores across all models (F1 in the 0.55-0.64 range) compared to the other two diseases. XGBoost performed best. This lines up with what showed up in the EDA - a meaningful chunk of the data had zero-as-missing values (Glucose, BloodPressure, Insulin, BMI) that had to be imputed, and it's a smaller, noisier dataset overall (768 rows, real-world clinical noise), so the ceiling on achievable accuracy is naturally lower here. Lower recall in particular means some true diabetes cases are being missed - worth discussing in an interview as a real-world tradeoff (in a health-risk model, missing positive cases is more costly than false alarms, so recall matters as much as accuracy).

**Kidney Disease**: Every model scored 98-100%. This needs an honest caveat rather than being presented as simply 'the best model' - a small (~400 row), fairly clean dataset with strongly separable classes (ckd cases in this dataset tend to have very distinct lab values from notckd cases) commonly produces near-perfect scores like this. It is not necessarily a sign of the model being 'better' than the other two - it's more a property of how separable this particular dataset is. This is a great thing to proactively bring up in an interview: knowing when very high accuracy is a red flag worth double-checking (e.g. via cross-validation or checking for data leakage) is a sign of ML maturity.

## What This Demonstrates (for CV / interview framing)

- Built a **multi-disease prediction system** with 3 independent pipelines, each handling different data quality issues (heart: text categoricals to encode; diabetes: hidden zero-as-missing values; kidney: heavy missing data + mixed categorical/numeric types)
- Ran a **controlled activation function experiment** (ReLU vs tanh vs sigmoid vs ELU) on identical architectures, not just used one activation function by default
- Compared **classical ML vs deep learning** honestly, including cases where classical ML won - shows real understanding rather than defaulting to 'deep learning is always better'
- Recognized when a near-perfect score (kidney) needs scrutiny rather than being reported at face value

## Next Steps (if extending further)

- Cross-validation (k-fold) instead of a single train/test split, especially for the small kidney dataset, to confirm the near-perfect score isn't due to an easy random split
- Hyperparameter tuning (GridSearchCV / Optuna) for the classical models
- SHAP values for model explainability - important in healthcare applications where predictions need to be justifiable
- Try recall-optimized thresholds for diabetes given the lower recall scores observed